# مناقصه‌یار · TenderYar v0.4 «Better search + TenderQA v0» — Course 4 project notebook
**Goal:** measure retrieval on **TenderQA v0** (60 questions with gold pages), then raise **recall@5 / MRR** with Persian clean-up,
BM25 + dense **hybrid** (RRF), multi-query, parent documents, a **Jalali-aware self-query** and re-ranking.

| Step | Course 4 lesson it re-uses | Cells | Visible result |
|---|---|---|---|
| Setup | restore v0.3, install, tests | 1–2 | `49 passed` |
| s1 Baseline + TenderQA v0 | W1 vector-store retriever, evaluation | 3–6 | dense recall@5 / MRR = the "from X" |
| s2 BM25 + hybrid | W1 BM25 + Query Fusion (RRF), W2 MMR | 7–9 | hybrid beats both |
| s3 Multi-query + parent | W1 MultiQuery, ParentDocument | 10–12 | 3 paraphrases · full «ماده ۳۰» |
| s4 Jalali self-query | W1 SelfQuery | 13–14 | 6/6 filters correct |
| s5 Re-rank + final bench | W1 retrievers by use case, W2 FAISS/Chroma | 15–18 | bar chart + RESULTS rows |
| Done | git tag v0.4 | 19 | tag pushed |

**MODE:** `"gemini"` needs `GOOGLE_API_KEY` in Colab Secrets. `"offline"` runs everything without a key (same plumbing, stand-in models).
Models: `gemini-3.8-flash` (main) · `gemini-3.5-flash-lite` (cheap: paraphrases, judge) · `gemini-2.5-flash` (fallback) · `models/gemini-embedding-2`.

## 0 · Setup — restore v0.3, add v0.4 files, run the tests

In [ ]:
# Cell 1 · install + restore the repo (Drive keeps it between sessions)
!pip install -q -U langchain-google-genai langchain-core langchain-community langchain-text-splitters langchain-chroma chromadb pymupdf rank_bm25 hazm sentence-transformers matplotlib pandas
import os, pathlib
from google.colab import drive
drive.mount("/content/drive")
HOME = pathlib.Path("/content/drive/MyDrive/tenderyar")       # the unzipped project folder
if not HOME.exists():
    from google.colab import files
    up = files.upload()                                        # choose tenderyar_c4_cpNN.zip
    !unzip -q -o "{list(up)[0]}" -d /content/drive/MyDrive/
os.chdir(HOME); print(os.getcwd())

In [ ]:
# Cell 2 · key + MODE + models + tests
MODE = "gemini"            # or "offline" (no key, reproduces docs/results_c4_offline.json)
if MODE == "gemini":
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
from tenderyar.config import LLM_MAIN, LLM_CHEAP, LLM_FALLBACK, EMBED_MODEL, RETRIEVER
from tenderyar.rag.index import HashEmbeddings, get_embeddings, build_index
from tenderyar.rag.qa import get_llm
EMB = get_embeddings(EMBED_MODEL if MODE == "gemini" else "offline")
EMB_RAW = EMB if MODE == "gemini" else HashEmbeddings(normalise=False)   # Gemini has no "clean-up off" switch
CHEAP = get_llm(LLM_CHEAP) if MODE == "gemini" else None                  # paraphrases + judge
print(MODE, LLM_MAIN, LLM_CHEAP, EMBED_MODEL)
!python -m pytest -q

## s1 · Baseline + TenderQA v0 (Module 1: retrievers + evaluation)

In [ ]:
# Cell 3 · load TenderQA v0 (60 rows) and count types
import json, collections, pandas as pd
from tenderyar.rag import bench as B, retrievers as T, metrics as M
QA = B.load_tenderqa("data/tenderqa/v0.jsonl")
print(len(QA), "questions ·", sum(bool(q["gold_pages"]) for q in QA), "answerable")
print(collections.Counter(q["type"] for q in QA))
pd.DataFrame(QA)[["id", "type", "question", "answer", "gold_pages"]].head(8)

In [ ]:
# Cell 4 · (optional, Gemini) 30 candidate questions from random chunks → CHECK BY HAND
import random, re
PROMPT = '''از متن زیر یک سؤال کوتاه فارسی و پاسخ دقیق آن بساز. فقط JSON: {{"question": "...", "answer": "...", "evidence": "عبارت دقیق از متن"}}
متن: {chunk}'''
if MODE == "gemini":
    random.seed(4); cands = []
    pages, chunks = B.load_package("P02")
    for ch in random.sample(chunks, 30):
        txt = M.msg_text(CHEAP.invoke(PROMPT.format(chunk=ch.page_content[:1200])))
        try:
            js = json.loads(txt[txt.index("{"): txt.rindex("}") + 1])
            cands.append({**js, "package": "P02", "page_guess": ch.metadata["page"]})
        except ValueError:
            pass
    pathlib.Path("data/tenderqa/candidates_gemini.jsonl").write_text("\n".join(json.dumps(c, ensure_ascii=False) for c in cands), encoding="utf-8")
    print(len(cands), "candidates → open data/tenderqa/candidates_gemini.jsonl, fix each one, write how many you fixed in FIXES (make_v0.py)")
else:
    print("offline: v0.jsonl already holds 30 checked template candidates (FIXES = 0)")

In [ ]:
# Cell 5 · dense baseline (v0.3 settings) → recall@5, MRR — this is your 'from X'
CFG = B.standard_configs(EMB, EMB_RAW, llm=CHEAP)
CACHE = {}
base = B.run({"1 dense (v0.3 baseline)": CFG["1 dense (v0.3 baseline)"]}, QA, cache=CACHE)
base

In [ ]:
# Cell 6 · the 10 worst questions and why they fail
det = base.attrs["detail"]["1 dense (v0.3 baseline)"]
qid = {q["id"]: q for q in QA}
worst = sorted(det, key=lambda r: (r["mrr"], r["id"]))[:10]
pd.DataFrame([{**w, "why": B.why_fails(qid[w["id"]]), "question": qid[w["id"]]["question"]} for w in worst])

## s2 · BM25 on clean Persian tokens, then hybrid (Module 1: BM25 + Query Fusion)

In [ ]:
# Cell 7 · tokens before / after search_normalize (3 hard examples)
from tenderyar.rag.clean import search_normalize, tokens
for h in ["تضمين شركت در مناقصه", "پیش‌پرداخت چند درصد است؟", "۵,۵۵۰,۰۰۰,۰۰۰ ریال"]:
    print(h.split(), "→", tokens(h))

In [ ]:
# Cell 8 · BM25 alone vs dense: where they disagree on the top-1 page
pages, chunks = CACHE["P01"]
vs = build_index(chunks, "p01", EMB)
bm, de = T.BM25(chunks, k=5), T.dense_retriever(vs, k=5)
pg = lambda docs: [d.metadata["page"] for d in docs][:3]
rows = [{"id": q["id"], "type": q["type"], "gold": q["gold_pages"], "bm25": pg(bm.invoke(q["question"])), "dense": pg(de.invoke(q["question"])), "q": q["question"]}
        for q in QA if q["package"] == "P01" and q["gold_pages"]]
hit1 = lambda l, g: bool(l) and l[0] in g
pd.DataFrame([r for r in rows if hit1(r["bm25"], r["gold"]) != hit1(r["dense"], r["gold"])])

In [ ]:
# Cell 9 · hybrid weights 0.3 / 0.5 / 0.7 and MMR on/off
names = ["1 dense (v0.3 baseline)", "dense + MMR", "2 BM25 (clean tokens)", "hybrid 0.3/0.7", "3 hybrid 0.5/0.5", "hybrid 0.7/0.3"]
grid = B.run({n: CFG[n] for n in names}, QA, cache=CACHE)
grid.drop(columns=["n"])

## s3 · Multi-query + parent document (Module 1: MultiQuery, ParentDocument retrievers)

In [ ]:
# Cell 10 · 3 paraphrases for «جریمه تأخیر چقدر است؟»
for p in T.paraphrases("جریمه تأخیر چقدر است؟", llm=CHEAP, n=3):
    print("•", p)
print(T.MQ_PROMPT[:120], "…")

In [ ]:
# Cell 11 · recall with / without multi-query (by question type)
mq = B.run({n: CFG[n] for n in ["3 hybrid 0.5/0.5", "4 hybrid + multi-query"]}, QA, cache=CACHE)
display(mq.drop(columns=["n"]))
pd.concat({n: B.by_type(mq, n) for n in mq.config}, axis=1)

In [ ]:
# Cell 12 · parent retriever: a small child hit returns the whole «ماده»
pr = T.parent_retriever(pages, k=3)
top = pr.invoke("جریمه تأخیر چقدر است؟")[0]
print("page", top.metadata["page"], "·", len(top.page_content), "chars"); print(top.page_content)

## s4 · Self-query with Jalali dates (Module 1: SelfQuery retriever)

In [ ]:
# Cell 13 · library of 60 tenders → structured query (semantic part + filter JSON) → results
import tempfile
from tenderyar.library import store
from tenderyar.library.cards import load_cards
from tenderyar.rag.splitter import persian_splitter
coll = store.get_collection(store.get_client(tempfile.mkdtemp())); sp = persian_splitter()
for c in load_cards("data/cards.jsonl"):
    store.upsert_tender(coll, c, sp.split_text(open(c["text_path"], encoding="utf-8").read()), EMB)
SQ = T.self_query(coll, llm=CHEAP, embeddings=EMB, k=5)        # dates: ALWAYS our pre-parser, never the LLM
q = "مناقصه‌های راه‌سازی اصفهان با مهلت قبل از ۱۵ آبان ۱۴۰۵"
s = SQ.structured(q); print(json.dumps(s, ensure_ascii=False, indent=1))
[(h["tender_id"], h["meta"]["province"], h["meta"]["deadline_jalali"]) for h in SQ.invoke(q)]

In [ ]:
# Cell 14 · 5 test phrasings: pre-parser vs LLM alone (today = 1405/08/20)
PH = ["مناقصه‌های راه‌سازی اصفهان با مهلت قبل از ۱۵ آبان ۱۴۰۵", "مناقصه آب بعد از ۱۵ آبان", "مناقصه‌های برق تا ۱۵ آبان",
      "مناقصه‌های همین هفته", "مناقصه‌های ماه آینده", "مهلت ۱۴۰۵/۰۸/۱۵"]
pd.DataFrame([{"q": p, "pre-parser": json.dumps(T.jalali_filter(p)[0]), "LLM alone": json.dumps(T.llm_date_filter(p, llm=CHEAP))} for p in PH])

## s5 · Re-rank and the final benchmark (Module 1 retrievers by use case · Module 2 vector stores)

In [ ]:
# Cell 15 · cross-encoder re-rank: top-20 → top-5
RR = T.get_reranker("BAAI/bge-reranker-v2-m3" if MODE == "gemini" else "offline")   # bge: ~2.3 GB, use a GPU runtime
hy20 = T.hybrid(T.BM25(chunks, k=20), T.dense_retriever(vs, k=20), weights=(0.5, 0.5), k=20)
q = "سقف جریمه تاخیر چند درصد مبلغ پیمان است؟"
top20 = hy20.invoke(q)
print("before:", [d.metadata["page"] for d in top20][:5]); print("after: ", [d.metadata["page"] for d in T.rerank(top20, q, 5, reranker=RR)])

In [ ]:
# Cell 16 · bench.run over the 6 spec configs + bar chart
CFG = B.standard_configs(EMB, EMB_RAW, llm=CHEAP, reranker=RR, grid=False)
df = B.run(CFG, QA, cache=CACHE)
B.plot(df, "docs/img/v0_4_recall.png")
from IPython.display import Image; display(df.drop(columns=["n"])); Image("docs/img/v0_4_recall.png")

In [ ]:
# Cell 17 · LLM-as-judge (0/1/2) on the answers of the best two configs
best2 = df.sort_values(["MRR", "recall@5"], ascending=False).config.head(2).tolist()
ans_llm = get_llm(LLM_MAIN) if MODE == "gemini" else None
from tenderyar.rag.qa import build_qa_chain
def gen(question, docs):
    if ans_llm is None: return B.extractive_answer(question, docs)
    ctx = "\n\n".join(f"[ص {d.metadata['page']}] {d.page_content}" for d in docs)
    return M.msg_text(ans_llm.invoke(f"فقط با تکیه بر متن زیر پاسخ کوتاه بده.\n{ctx}\nسؤال: {question}"))
jd = B.run({n: CFG[n] for n in best2}, QA, cache=CACHE, answer_fn=gen, judge_llm=CHEAP)
jd[["config", "recall@5", "MRR", "judge (0–2)"]]

In [ ]:
# Cell 18 · RESULTS.md rows + choose the default config
sfx = " · Gemini" if MODE == "gemini" else " (offline)"
print(B.results_rows(df, sfx))          # paste into docs/RESULTS.md · v0.4
print("current default:", RETRIEVER)
# if Gemini MQ / bge re-rank win: edit tenderyar/config.py → RETRIEVER["multi_query"]=True / ["rerank"]=True

## Done — tests, commit, tag v0.4

In [ ]:
# Cell 19 · tests + commit + tag
!python -m pytest -q
!git add -A && git commit -qm "v0.4 better search + TenderQA v0" && git tag -f v0.4 && git log --oneline -1
# push (needs a GitHub token in Colab Secrets GH_TOKEN):
# from google.colab import userdata; tok = userdata.get("GH_TOKEN")
# !git push https://{tok}@github.com/<you>/tenderyar.git main --tags